### 06 - Modele lineaire d'attrition
#### HumanForYou - Attrition ML

Objectif: entrainer un modele lineaire (Logistic Regression) sur les donnees preparees en 05, elles-memes derivees de 04.

- **Entrees**: `data/processed/attrition_train_prepared.csv`, `data/processed/attrition_test_prepared.csv`
- **Sorties**: `data/processed/attrition_linear_metrics.csv`, `data/processed/attrition_linear_test_predictions.csv`, `data/processed/attrition_linear_coefficients.csv`

#### 1. Imports

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, roc_curve

#### 2. Chargement

In [ ]:
PROCESSED_DIR = os.path.join('..', 'data', 'processed')
train_path = os.path.join(PROCESSED_DIR, 'attrition_train_prepared.csv')
test_path = os.path.join(PROCESSED_DIR, 'attrition_test_prepared.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

X_train = train_df.drop(columns=['Attrition']).copy()
y_train = train_df['Attrition'].astype(int).copy()
X_test = test_df.drop(columns=['Attrition']).copy()
y_test = test_df['Attrition'].astype(int).copy()

print(f'X_train: {X_train.shape} | X_test: {X_test.shape}')

#### 3. Entrainement et evaluation

In [ ]:
model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42)
model.fit(X_train, y_train)

y_proba = model.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

metrics_df = pd.DataFrame([
    {
        'model': 'LogisticRegression_balanced',
        'accuracy_test': float(accuracy_score(y_test, y_pred)),
        'precision_test': float(precision_score(y_test, y_pred, zero_division=0)),
        'recall_test': float(recall_score(y_test, y_pred, zero_division=0)),
        'f1_test': float(f1_score(y_test, y_pred, zero_division=0)),
        'roc_auc_test': float(roc_auc_score(y_test, y_proba)),
    }
])

predictions_df = pd.DataFrame({
    'y_true': y_test,
    'y_proba_logistic': y_proba,
    'y_pred_logistic': y_pred,
})

coef_df = pd.DataFrame({
    'feature': X_train.columns,
    'coefficient': model.coef_.ravel(),
}).sort_values('coefficient', key=np.abs, ascending=False)

metrics_df

#### 4. Export + figures

In [ ]:
FIG_DIR = os.path.join('..', 'reports', 'figures')
os.makedirs(FIG_DIR, exist_ok=True)

metrics_path = os.path.join(PROCESSED_DIR, 'attrition_linear_metrics.csv')
preds_path = os.path.join(PROCESSED_DIR, 'attrition_linear_test_predictions.csv')
coef_path = os.path.join(PROCESSED_DIR, 'attrition_linear_coefficients.csv')

metrics_df.to_csv(metrics_path, index=False)
predictions_df.to_csv(preds_path, index=False)
coef_df.to_csv(coef_path, index=False)

top_coef = coef_df.head(15).sort_values('coefficient')
plt.figure(figsize=(9, 6))
plt.barh(top_coef['feature'], top_coef['coefficient'])
plt.title('Top 15 coefficients (Logistic Regression)')
plt.tight_layout()
coef_fig_path = os.path.join(FIG_DIR, 'attrition_linear_top_coefficients.png')
plt.savefig(coef_fig_path, dpi=300)
plt.close()

fpr, tpr, _ = roc_curve(y_test, y_proba)
plt.figure(figsize=(7, 5))
plt.plot(fpr, tpr, label='Logistic Regression')
plt.plot([0, 1], [0, 1], '--', color='gray')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve - Attrition')
plt.legend()
plt.tight_layout()
roc_fig_path = os.path.join(FIG_DIR, 'attrition_linear_roc_curve.png')
plt.savefig(roc_fig_path, dpi=300)
plt.close()

print(f'Saved: {metrics_path}')
print(f'Saved: {preds_path}')
print(f'Saved: {coef_path}')
print(f'Saved: {coef_fig_path}')
print(f'Saved: {roc_fig_path}')